In [1]:
import numpy as np
import pandas as pd
import os

def calculate_momentum_ratios(returns_file, benchmark_file):
    """
    Calculate rolling returns and volatility ratios for a momentum strategy 
    given a returns file and benchmark file.

    Parameters
    ----------
    returns_file : str
        Path to the Excel file containing 'Date' and 'Buy_Hold_Value' columns.
    benchmark_file : str
        Path to the Excel file containing 'Date' and 'Price' columns.

    Returns
    -------
    str
        Output file name (without full path).
    """

    # --- Load benchmark file ---
    benchmark = (
        pd.read_excel(benchmark_file)
        .rename(columns={'Buy_Hold_Value': 'benchmark_value'})
    )
    benchmark['pct_change_benchmark'] = benchmark['benchmark_value'].pct_change()
    benchmark['Rolling_Returns_252_benchmark'] = (
        (1 + benchmark['pct_change_benchmark']).rolling(252).apply(np.prod, raw=True) - 1
    )

    # --- Load returns file ---
    df = (
        pd.read_excel(returns_file, usecols=['Date', 'Buy_Hold_Value'])
        .rename(columns={'Buy_Hold_Value': 'value'})
    )

    # --- Aggregate and calculate rolling returns for portfolio ---
    grouped_df = df.groupby('Date', as_index=False)['value'].sum()
    grouped_df['pct_change_value'] = grouped_df['value'].pct_change()
    grouped_df['Rolling_Returns_252'] = (
        (1 + grouped_df['pct_change_value']).rolling(252).apply(np.prod, raw=True) - 1
    )

    # --- Merge with benchmark ---
    merged_df = grouped_df.merge(benchmark, on='Date', how='left')
    merged_df['Rolling_Returns_alpha_252'] = merged_df['Rolling_Returns_252'] - merged_df['Rolling_Returns_252_benchmark']
    merged_df['Rolling_Returns_alpha_cumsum_252'] = merged_df['Rolling_Returns_alpha_252'].cumsum()


    # --- Calculate rolling annualized standard deviations ---
    windows = [21, 63, 126, 252, 504]
    for w in windows:
        merged_df[f'SD_{w}'] = (
            merged_df['pct_change_value'].rolling(w).std() * np.sqrt(252)
        )
    merged_df['Sharpe_252'] = (merged_df['Rolling_Returns_252'] - 0.05) / merged_df['SD_252']

    # --- Generate output file name ---
    base_name = os.path.basename(returns_file)
    output_file = base_name.replace('returns', 'ratios')

    # --- Save to Excel (you can prepend your own path when calling) ---
    output_path = "Ratios And KPIs\\"
    merged_df.to_excel(output_path + output_file, index=False)

    return output_file


In [2]:
#Benchmarks

In [3]:
output_name = calculate_momentum_ratios(
    r"Trials\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx",
    r"Trials\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx"
)
print("✅ Ratios file name:", output_name)

✅ Ratios file name: nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_ratios.xlsx


In [4]:
output_name = calculate_momentum_ratios(
    r"Trials\Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx",
    r"Trials\nse500_Nifty_500_2025_Apr_nse500_nse500_nse500_nse500_nse500_returns.xlsx"
)
print("✅ Ratios file name:", output_name)

✅ Ratios file name: Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_ratios.xlsx
